# 🤟 KZH Sign Language Translator — Обучение модели

**Автор:** Ученик 8 класса Aqbobek International School

**Цель:** Обучить нейросеть распознавать жесты казахского жестового языка (КЖЯ)

**Этот ноутбук предназначен для Google Colab** — бесплатной платформы с GPU от Google.

---

## 📚 План ноутбука:
1. Установка зависимостей
2. Загрузка датасета
3. Создание модели
4. Обучение
5. Графики обучения
6. Сохранение модели
7. Тест на одном примере

---
## Ячейка 1: Установка зависимостей

Устанавливаем библиотеки, которых нет в Colab по умолчанию.
TensorFlow уже предустановлен в Colab, но мы устанавливаем конкретную версию для воспроизводимости.

In [ ]:
# === Ячейка 1: Установка зависимостей ===
# mediapipe и opencv-python нужны для обработки видео и извлечения координат руки
# tensorflow уже есть в Colab, но устанавливаем конкретную версию
!pip install tensorflow==2.15.0 mediapipe==0.10.8 opencv-python==4.8.1.78

# Проверяем, что всё установилось
import tensorflow as tf
import numpy as np
print(f"✅ TensorFlow версия: {tf.__version__}")
print(f"✅ NumPy версия: {np.__version__}")
print(f"✅ GPU доступен: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Ячейка 2: Загрузка датасета

Загружаем файл `dataset.npz`, который содержит:
- `X` — массив координат руки, форма (N, 30, 63)
- `y` — метки классов, форма (N,)
- `label_map` — словарь {жест: число}

Этот файл создаётся локально через `extract_landmarks.py` + `normalize.py`.

In [ ]:
# === Ячейка 2: Загрузка датасета ===
import json
import numpy as np

# Способ 1: Загрузка через интерфейс Colab (кнопка "Выберите файл")
# google.colab.files.upload() открывает диалог выбора файла
# на вашем компьютере и загружает его в Colab.
try:
    from google.colab import files
    print("Загрузите файл dataset.npz:")
    uploaded = files.upload()
    dataset_path = list(uploaded.keys())[0]
except ImportError:
    # Способ 2: Если запускаем локально (не в Colab)
    dataset_path = "dataset.npz"
    print(f"Загружаю локально: {dataset_path}")

# Загружаем данные из .npz файла
# .npz — это архив из нескольких numpy-массивов
data = np.load(dataset_path, allow_pickle=True)

X = data['X']           # Координаты руки: (кол-во примеров, 30, 63)
y = data['y']           # Метки классов: (кол-во примеров,)
label_map = data['label_map'].item()  # Словарь {жест: число}

# Создаём обратный словарь: {число: жест}
# Нужен, чтобы превратить предсказание модели (число) обратно в название жеста
reverse_label_map = {v: k for k, v in label_map.items()}

num_classes = len(label_map)

print(f"\n✅ Датасет загружен!")
print(f"   X.shape = {X.shape}  (примеры, кадры, координаты)")
print(f"   y.shape = {y.shape}  (метки)")
print(f"   Классов: {num_classes}")
print(f"   Жесты: {label_map}")

---
## Ячейка 3: Создание модели

Создаём нейросеть с архитектурой:
1. **Masking** — игнорируем padding-нули
2. **Bidirectional LSTM (64)** — читаем последовательность в обоих направлениях
3. **Dropout (0.3)** — защита от переобучения
4. **Bidirectional LSTM (32)** — глубокий анализ
5. **Dense (32, relu)** — финальные признаки
6. **Dense (N, softmax)** — вероятности для N жестов

In [ ]:
# === Ячейка 3: Создание модели ===
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Masking, LSTM, Bidirectional, Dropout, Dense
)

# Параметры модели
SEQUENCE_LENGTH = X.shape[1]   # 30 кадров
NUM_FEATURES = X.shape[2]      # 63 координаты (21 точка × 3)

print(f"Параметры:")
print(f"  Кадров в последовательности: {SEQUENCE_LENGTH}")
print(f"  Признаков на кадр:           {NUM_FEATURES}")
print(f"  Количество классов (жестов):  {num_classes}")

# Строим модель послойно
model = Sequential(name='KZH_Sign_Classifier')

# Слой 1: Masking — игнорируем padding-нули
# Если все 63 числа в кадре = 0, этот кадр пропускается
model.add(Masking(mask_value=0.0, input_shape=(SEQUENCE_LENGTH, NUM_FEATURES)))

# Слой 2: Bidirectional LSTM (64 нейрона)
# LSTM — нейросеть с «памятью», читает последовательность кадров
# Bidirectional — читаем ВПЕРЁД И НАЗАД, получаем больше контекста
# return_sequences=True — передаём всю последовательность в следующий LSTM
model.add(Bidirectional(LSTM(64, return_sequences=True)))

# Слой 3: Dropout (30%)
# Выключаем 30% нейронов случайным образом при обучении,
# чтобы модель не «зазубривала» данные
model.add(Dropout(0.3))

# Слой 4: Второй Bidirectional LSTM (32 нейрона)
# Второй слой LSTM для более глубокого анализа паттернов
model.add(Bidirectional(LSTM(32)))

# Слой 5: Dense (32) с активацией ReLU
# Полносвязный слой, извлекает финальные признаки жеста
model.add(Dense(32, activation='relu'))

# Слой 6: Выходной Dense (num_classes) с softmax
# softmax превращает выход в вероятности (сумма = 1.0)
model.add(Dense(num_classes, activation='softmax'))

# Компиляция модели
# adam — адаптивный оптимизатор, автоматически подбирает скорость обучения
# sparse_categorical_crossentropy — функция ошибки для классификации
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Выводим архитектуру
model.summary()

---
## Ячейка 4: Обучение модели

Настраиваем процесс обучения:
- **epochs=50** — максимум 50 проходов по всем данным
- **batch_size=16** — обрабатываем 16 примеров за раз
- **validation_split=0.2** — 20% данных для проверки
- **EarlyStopping** — остановка, если модель не улучшается 10 эпох

In [ ]:
# === Ячейка 4: Обучение ===
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# EarlyStopping — если модель не улучшается 10 эпох подряд — стоп.
# restore_best_weights=True — возвращаемся к ЛУЧШЕМУ состоянию
early_stop = EarlyStopping(
    monitor='val_loss',         # Следим за ошибкой на валидации
    patience=10,                # Ждём 10 эпох без улучшения
    restore_best_weights=True,  # Восстанавливаем лучшие веса
    verbose=1                   # Выводим сообщение при срабатывании
)

# ModelCheckpoint — сохраняем модель каждый раз, когда она становится лучше
checkpoint = ModelCheckpoint(
    'best_model.h5',            # Путь для сохранения
    monitor='val_loss',
    save_best_only=True,        # Только лучшую версию
    verbose=1
)

print("🚀 Начинаем обучение...\n")

# model.fit() — главная функция обучения
# Модель смотрит на X (координаты), сравнивает своё предсказание с y (правильным ответом),
# и корректирует свои веса (parameters), чтобы в следующий раз ошибаться меньше.
history = model.fit(
    X, y,
    epochs=50,              # Максимум 50 проходов
    batch_size=16,          # 16 примеров за раз
    validation_split=0.2,   # 20% для проверки
    callbacks=[early_stop, checkpoint],
    verbose=1               # Прогресс-бар
)

print(f"\n✅ Обучение завершено!")
print(f"   Точность (train): {history.history['accuracy'][-1]:.4f}")
print(f"   Точность (val):   {history.history['val_accuracy'][-1]:.4f}")

---
## Ячейка 5: Графики обучения

Два графика показывают, КАК модель училась:
- **Accuracy** — доля правильных ответов (должна РАСТИ)
- **Loss** — ошибка модели (должна ПАДАТЬ)

Если линии train и val расходятся — это **переобучение**.

In [ ]:
# === Ячейка 5: Графики ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Точность (Accuracy)
axes[0].plot(history.history['accuracy'], label='Обучение', linewidth=2, color='#2196F3')
axes[0].plot(history.history['val_accuracy'], label='Валидация', linewidth=2, color='#FF9800')
axes[0].set_title('Точность модели (Accuracy)', fontsize=14)
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# График 2: Ошибка (Loss)
axes[1].plot(history.history['loss'], label='Обучение', linewidth=2, color='#2196F3')
axes[1].plot(history.history['val_loss'], label='Валидация', linewidth=2, color='#FF9800')
axes[1].set_title('Ошибка модели (Loss)', fontsize=14)
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_graphs.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Графики сохранены в training_graphs.png")

---
## Ячейка 6: Сохранение модели

Сохраняем обученную модель и маппинг меток, чтобы скачать на компьютер
и использовать в Streamlit-приложении.

In [ ]:
# === Ячейка 6: Сохранение модели ===
import json

# Сохраняем модель в формате HDF5 (.h5)
# Этот файл содержит архитектуру + обученные веса
model.save('sign_model.h5')
print("✅ Модель сохранена: sign_model.h5")

# Сохраняем маппинг меток в JSON
# {"привет": 0, "спасибо": 1, ...}
with open('label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print("✅ Маппинг меток сохранён: label_map.json")

# Скачиваем файлы на компьютер
try:
    from google.colab import files
    files.download('sign_model.h5')
    files.download('label_map.json')
    files.download('training_graphs.png')
    print("\n📥 Файлы скачиваются...")
except ImportError:
    print("\nЗапуск не в Colab. Файлы сохранены локально.")

---
## Ячейка 7: Тест на одном примере

Берём один пример из датасета и проверяем, правильно ли модель его классифицирует.

In [ ]:
# === Ячейка 7: Тест на одном примере ===
import numpy as np

# Выбираем случайный пример из датасета
test_idx = np.random.randint(0, len(X))
test_sample = X[test_idx:test_idx+1]  # Форма (1, 30, 63) — один пример
true_label = y[test_idx]
true_name = reverse_label_map[true_label]

# Предсказание модели
# model.predict() возвращает вероятности для КАЖДОГО класса
predictions = model.predict(test_sample, verbose=0)

# np.argmax находит индекс максимальной вероятности
predicted_label = np.argmax(predictions[0])
predicted_name = reverse_label_map[predicted_label]
confidence = predictions[0][predicted_label] * 100

print(f"🧠 Тест модели:")
print(f"   Правильный ответ:   {true_name} (код {true_label})")
print(f"   Предсказание:     {predicted_name} (код {predicted_label})")
print(f"   Уверенность:      {confidence:.1f}%")
print(f"   Результат:         {'✅ ПРАВИЛЬНО' if predicted_label == true_label else '❌ ОШИБКА'}")

# Выводим вероятности для всех классов
print(f"\n   Вероятности:")
for idx, prob in enumerate(predictions[0]):
    name = reverse_label_map[idx]
    bar = '█' * int(prob * 30)
    marker = ' ← предсказание' if idx == predicted_label else ''
    print(f"     {name:>15}: {bar} {prob*100:.1f}%{marker}")